In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import numpy as np

data = np.array([
   
    [29.4, 10.7, 0.16002],[25.6, 14.3, 0.14664],[27.9, 11.8, 0.15332],
    [28.3, 13.6, 0.15416],[26.7, 10.2, 0.15155],[30.0, 15.0, 0.16024],
    [25.1, 12.9, 0.14564],[29.8, 14.7, 0.16098],[28.6, 11.1, 0.15548],
    [27.0, 13.2, 0.15023],[26.1, 7.8, 0.15074],[29.8, 5.4, 0.16817],
    [27.0, 9.1, 0.15247],[25.9, 6.6, 0.15125],[28.7, 8.9, 0.15782],
    [26.5, 5.7, 0.15364],[29.1, 9.7, 0.16073],[27.6, 6.2, 0.15588],
    [25.3, 7.5, 0.14859],[28.9, 8.1, 0.15977],

    [21.7, 12.4, 0.1387],[24.8, 10.6, 0.14532],[20.9, 15.3, 0.13697],
    [22.5, 19.7, 0.1387],[23.9, 11.2, 0.14193],[21.3, 16.8, 0.13669],
    [24.1, 13.5, 0.14271],[20.6, 17.1, 0.13543],[22.8, 14.9, 0.13893],
    [21.1, 10.3, 0.13794],[24.6, 18.2, 0.14274],[23.0, 12.7, 0.13986],

    [22.2, 15.8, 0.13737],
    [24.3, 11.9, 0.14266],
    [21.9, 17.6, 0.13684],
    [23.5, 13.1, 0.14154],
    [20.8, 16.4, 0.13729],
    [24.0, 14.2, 0.1424],
])

X = data[:, :2]
y = data[:, 2]

In [3]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=200, random_state=42)
rf.fit(X, y)

RandomForestRegressor(n_estimators=200, random_state=42)

In [4]:
import random

alpha_min, alpha_max = X[:,0].min(), X[:,0].max()
phi_min, phi_max = X[:,1].min(), X[:,1].max()

def fitness(ind):
    return rf.predict([ind])[0]   # minimizing Cd

def create_population(size):
    return [[
        random.uniform(alpha_min, alpha_max),
        random.uniform(phi_min, phi_max)
    ] for _ in range(size)]

def select(pop):
    return sorted(pop, key=fitness)[:len(pop)//2]

# Improved crossover
def crossover(p1, p2):
    alpha = random.random()
    return [
        alpha*p1[0] + (1-alpha)*p2[0],
        alpha*p1[1] + (1-alpha)*p2[1]
    ]

# Mutation + bounds
def mutate(ind):
    if random.random() < 0.2:
        ind[0] += random.uniform(-0.5, 0.5)
        ind[1] += random.uniform(-0.5, 0.5)

    # Clamp within bounds
    ind[0] = max(alpha_min, min(alpha_max, ind[0]))
    ind[1] = max(phi_min, min(phi_max, ind[1]))

    return ind

# The GA loop 
def run_ga(generations=50, pop_size=30):
    population = create_population(pop_size)

    for _ in range(generations):
        population = sorted(population, key=fitness)

        # Elitism (keep best)
        best = population[0]

        selected = select(population)
        next_gen = [best] + selected.copy()

        while len(next_gen) < pop_size:
            p1, p2 = random.sample(selected, 2)
            child = crossover(p1, p2)
            child = mutate(child)
            next_gen.append(child)

        population = next_gen

    best = min(population, key=fitness)
    return best, fitness(best)

best_global = None
best_score = float("inf")

for i in range(10):
    sol, score = run_ga()

    print(f"Run {i+1}: alpha={sol[0]:.3f}, phi={sol[1]:.3f}, Cd={score:.6f}")

    if score < best_score:
        best_score = score
        best_global = sol

print("\n BEST OVERALL SOLUTION:")
print("alpha, phi =", best_global)
print("Minimum Cd =", best_score)

Run 1: alpha=20.600, phi=17.145, Cd=0.135950
Run 2: alpha=20.692, phi=17.015, Cd=0.135950
Run 3: alpha=20.685, phi=17.023, Cd=0.135950
Run 4: alpha=20.650, phi=17.018, Cd=0.135950
Run 5: alpha=20.600, phi=17.154, Cd=0.135950
Run 6: alpha=20.600, phi=17.031, Cd=0.135950
Run 7: alpha=20.698, phi=17.145, Cd=0.135950
Run 8: alpha=20.600, phi=17.043, Cd=0.135950
Run 10: alpha=20.691, phi=17.087, Cd=0.135950

 BEST OVERALL SOLUTION:
alpha, phi = [np.float64(20.6), np.float64(17.144608245124658)]
Minimum Cd = 0.13595039999999992
